<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 3: Case C and Decision Domains

**The snack order must distinguish fruit measured by weight from boxes ordered as whole counts.**

Part 2 showed how requirements determine feasibility. This part formulates the school-event snack order from 01-2 and changes only which decision types are included.

### 1 · Carry forward the snack-order case

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_c_snacks.png" alt="Whole snack boxes next to loose fruit on a kitchen scale, illustrating counting boxes and weighing fruit." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case C from 01-2: the box count \(z\) and fruit amount \(p\) form \(x=[z,p]^{\mathsf T}\). Their allowed values give the decision a mixed domain.

Let \(z\) be the number of 8-portion snack boxes and let \(p\) be the kilograms of fruit, which provide 4 portions per kilogram. Boxes cost \$12 each, and fruit costs \$7 per kilogram. The event needs at least 30 portions, with at most 5 boxes and 8 kg of fruit available.

The decision domain must record that \(z\) is a whole count while \(p\) is a measured amount.

### 2 · Formulate the combined order

Use the decision vector

> $\displaystyle x=\begin{bmatrix}z\\p\end{bmatrix}\in\{0,1,\ldots,5\}\times[0,8].$

The direct response is \(y=[C(x),P(x)]^{\mathsf T}\), where cost is \(C(x)=7p+12z\) and portions are \(P(x)=4p+8z\). The formulation is

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad f(y)=C(x)=7p+12z$
>
> $\displaystyle \text{subject to}\quad g_1(x)=30-P(x)\le0,$
>
> $\displaystyle x\in\{0,1,\ldots,5\}\times[0,8].$

The portion constraint determines whether an order is sufficient. The domain determines whether each numerical value is an allowed action.

### 3 · Compare continuous, discrete, and mixed variants

| Variant | Decision domain | Type |
|:---|:---|:---|
| Fruit only | \(p\in[0,8]\) | Continuous |
| Boxes only | \(z\in\{0,1,\ldots,5\}\) | Discrete |
| Fruit and boxes | \((z,p)\in\{0,1,\ldots,5\}\times[0,8]\) | Mixed |

A continuous variable may take any real value in its interval. A discrete variable may take only listed or countable values. A mixed decision vector contains both.

Each variant minimizes the cost of the included item types and requires at least 30 portions. Only the allowed decision values change.

The calculation below keeps whole box counts and a 0.25-kg fruit search grid.

In [ ]:
import numpy as np

MAX_FRUIT = 8.0
MAX_BOXES = 5
REQUIRED_PORTIONS = 30.0


def evaluate_order(box_count, fruit_kg, required_portions=REQUIRED_PORTIONS):
    decision = (float(box_count), float(fruit_kg))
    cost = 12.0 * decision[0] + 7.0 * decision[1]
    portions = 8.0 * decision[0] + 4.0 * decision[1]
    feasible = (
        0 <= decision[0] <= MAX_BOXES
        and decision[0].is_integer()
        and 0.0 <= decision[1] <= MAX_FRUIT
        and portions >= float(required_portions)
    )
    return {"x": decision, "cost": cost, "portions": portions, "feasible": feasible}

Each vertical segment on the left belongs to one whole box count \(z\). Fruit amount \(p\) can vary continuously along it. Gray segments provide too few portions, while teal segments meet the requirement. A point between columns would require a fractional box.

On the right, each teal bar is the least feasible cost sampled at that box count. The orange point shows the current order's cost. The initial order \((z,p)=(2,2)\) is cheap but supplies only 24 of the required 30 portions. The gold star marks the best feasible candidate on the 0.25-kg fruit grid.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_c_formulation_graph.png" alt="Vertical feasible fruit-amount segments at integer box counts link a snack order to minimum feasible costs and the selected grid order." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
import sys
import matplotlib


def _pyplot(*, interactive=False):
    """Use the course's widget-backend fallback outside the browser runtime."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
PURPLE = "#8856a7"
GRAY = "#a6a6a6"
GOLD = "#f6c945"


def case_canvas(title, controls, *, panels=2, interactive=False):
    """Create a shared layout; controls are (name, label, low, high, value, step, color)."""
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(
        1, panels, figsize=(12.4 if panels == 2 else 14.0, 7.4 if interactive else 5.3)
    )
    figure.suptitle(title, y=0.985, fontsize=14, fontweight="bold")
    status = figure.text(0.5, 0.918, "", ha="center", va="center", fontsize=11)
    footer = figure.text(0.5, 0.045 if not interactive else 0.27, "",
                         ha="center", va="center", fontsize=9)
    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider

        figure.subplots_adjust(left=0.075, right=0.97,
                               bottom=0.41 if panels == 3 else 0.37, top=0.83, wspace=0.38)
        positions = np.linspace(0.19, 0.065, max(len(controls), 2))
        for position, (name, label, low, high, value, step, color) in zip(positions, controls):
            slider_axis = figure.add_axes([0.28, position, 0.61, 0.026])
            sliders[name] = Slider(slider_axis, label, low, high, valinit=value,
                                   valstep=step, valfmt="%1.0f" if step >= 1 else "%1.2f",
                                   color=color, initcolor=color)
    figure._case_sliders = sliders
    figure._case_state = {}
    return plt, figure, axes, sliders, status, footer


def finish_case(plt, figure, axes, sliders, refresh, *, interactive=False):
    """Connect controls and keep widgets and evaluated records alive on the figure."""
    for axis in axes:
        axis.grid(alpha=0.25)
    for slider in sliders.values():
        slider.on_changed(refresh)
    figure._case_refresh = refresh
    refresh()
    if not interactive:
        figure.tight_layout(rect=(0.015, 0.09, 0.985, 0.96))
    plt.show()
    if not interactive:
        plt.close(figure)
    return figure


def show_snack_formulation(decision=(2.0, 2.0), required_portions=REQUIRED_PORTIONS, *, interactive=False):
    controls = [
        ("boxes", "Decision: boxes z", 0, MAX_BOXES, decision[0], 1, BLUE),
        ("fruit", "Decision: fruit p (kg)", 0.0, MAX_FRUIT, decision[1], 0.25, BLUE),
        ("portions", "Requirement: portions needed", 16, 48, required_portions, 1, TEAL),
    ]
    plt, figure, axes, sliders, status, footer = case_canvas(
        "Case C · Whole boxes and fruit amounts create a mixed decision domain",
        controls, interactive=interactive,
    )
    fruit_grid = np.arange(0.0, MAX_FRUIT + 0.125, 0.25)
    infeasible_segments, feasible_segments = [], []
    for z in range(MAX_BOXES + 1):
        minimum_fruit = max(0.0, (REQUIRED_PORTIONS - 8.0 * z) / 4.0)
        low_line, = axes[0].plot([z, z], [0, minimum_fruit], color=GRAY, linewidth=5,
                                 label="Too few portions" if z == 0 else None)
        high_line, = axes[0].plot([z, z], [minimum_fruit, MAX_FRUIT], color=TEAL, linewidth=5,
                                  label="Feasible fruit amounts" if z == 0 else None)
        infeasible_segments.append(low_line)
        feasible_segments.append(high_line)
    portion_line, = axes[0].plot([], [], ":", color=BLUE, linewidth=1.4,
                                 label="Required-portion boundary")
    best_domain = axes[0].scatter([], [], marker="*", color=GOLD, edgecolor="black", s=230,
                                   zorder=5, label="Best grid candidate")
    domain_marker = axes[0].scatter([], [], color=ORANGE, edgecolor="black", s=110, zorder=6)
    axes[0].set(xlabel="Snack-box count z (whole boxes)", ylabel="Fruit amount p (kg)",
                xticks=range(MAX_BOXES + 1), xlim=(-0.45, MAX_BOXES + 0.45),
                ylim=(-0.3, 11.5), title="Each vertical segment allows continuous p")
    axes[0].legend(fontsize=8, loc="upper right")
    counts = np.arange(MAX_BOXES + 1)
    minimum_cost_bars = axes[1].bar(counts, np.zeros(len(counts)), color=TEAL, alpha=0.55,
                                   label="Least feasible grid cost at this z")
    best_cost = axes[1].scatter([], [], marker="*", color=GOLD,
                                edgecolor="black", s=230, zorder=5, label="Best grid candidate")
    cost_marker = axes[1].scatter([], [], color=ORANGE, edgecolor="black", s=110,
                                   zorder=6, label="Current order")
    axes[1].set(xlabel="Snack-box count z (whole boxes)", ylabel="Order cost C ($)",
                xticks=counts, xlim=(-0.45, MAX_BOXES + 0.45), ylim=(0, 130),
                title="Low cost alone does not guarantee enough portions")
    axes[1].legend(fontsize=8, loc="upper left")
    footer.set_text("The star uses a 0.25-kg search grid. The fruit slider samples a continuous variable; boxes remain whole counts.")

    def refresh(_=None):
        values = (sliders["boxes"].val, sliders["fruit"].val) if sliders else decision
        required = sliders["portions"].val if sliders else required_portions
        current = evaluate_order(*values, required_portions=required)
        records = [evaluate_order(z, p, required_portions=required)
                   for z in range(MAX_BOXES + 1) for p in fruit_grid]
        feasible = [r for r in records if r["feasible"]]
        best = min(feasible, key=lambda r: r["cost"])
        best_by_count = []
        for z, low_line, high_line, bar in zip(counts, infeasible_segments, feasible_segments,
                                               minimum_cost_bars):
            minimum_fruit = max(0.0, (required - 8.0 * z) / 4.0)
            low_line.set_ydata([0, min(minimum_fruit, MAX_FRUIT)])
            high_line.set_ydata([minimum_fruit, MAX_FRUIT]
                                if minimum_fruit <= MAX_FRUIT else [np.nan, np.nan])
            choices = [r for r in feasible if r["x"][0] == z]
            least_cost = min(choices, key=lambda r: r["cost"]) if choices else None
            best_by_count.append(least_cost)
            bar.set_height(least_cost["cost"] if least_cost is not None else np.nan)
        portion_line.set_data([0, required / 8], [required / 4, 0])
        best_domain.set_offsets([best["x"]])
        best_cost.set_offsets([[best["x"][0], best["cost"]]])
        domain_marker.set_offsets([current["x"]])
        cost_marker.set_offsets([[current["x"][0], current["cost"]]])
        eligibility = "FEASIBLE" if current["feasible"] else "REJECTED"
        status.set_text(f"Current z = {values[0]:.0f}, p = {values[1]:.2f} kg"
                        f"   |   Portions = {current['portions']:.0f} / {required:.0f} required"
                        f"   |   Cost = {current['cost']:.2f} dollars   |   {eligibility}")
        status.set_color(TEAL if current["feasible"] else "#555555")
        figure._case_state.update(current=current, best=best, best_by_count=best_by_count,
                                   required_portions=required)
        figure.canvas.draw_idle()

    return finish_case(plt, figure, axes, sliders, refresh, interactive=interactive)

In [ ]:
static_figure = show_snack_formulation(decision=(2.0, 2.0))

The box trackbar moves in whole counts, while the fruit trackbar samples a continuous amount in 0.25-kg increments. The teal trackbar changes the required number of portions.

For this controlled variation, \(P_{\mathrm{req}}\) replaces the baseline requirement 30:

> $\displaystyle g_1(x;P_{\mathrm{req}})=P_{\mathrm{req}}-(8z+4p)\le0.$

A higher requirement shifts the feasible segments upward and can change the selected order. A missing cost bar means that no allowed fruit amount can meet the requirement at that box count.

In [ ]:
formulation_explorer = show_snack_formulation(
    decision=(2.0, 2.0), interactive=True
)

The result is the best candidate on the stated fruit grid. The box domain itself is genuinely discrete, but the 0.25 kg fruit grid is only a search choice. Fruit remains a continuous decision in the formulation.

### 4 · Keep domain and search method separate

| Statement | What it describes |
|:---|:---|
| \(p\in[0,8]\) | The real fruit decision is continuous |
| Evaluate \(p=0,0.25,\ldots,8\) | The demonstration samples a finite search grid |
| \(z\in\{0,1,\ldots,5\}\) | The real box decision is discrete |

The combined case is a **generally constrained, mixed, single-objective, linear, direct algebraic, deterministic optimization problem**. Because its objective and constraints are linear and it contains an integer variable, it is a mixed-integer linear program (MILP).

### Takeaway

Classify decision values from the real action, not from the search code:

> **measured amount → continuous · whole count or named option → discrete · both in one decision vector → mixed**

A finite grid gives the best sampled candidate, not a guaranteed continuous optimum. Part 4 keeps continuous decisions and changes the number of objectives.